# 10 — Post-hoc Confidence Calibration: Leakage-Safe OOF (development)

**Dataset:** `validated_v007`; **input:** `feature_assembly_v2_repair1` (Stage 05 repair-aware).

Bu notebook **final test değildir**. Yalnız `exclude_from_final_test == 0` olan `CLEAN_DEV` kayıtlarında model başına bağımsız 5-fold stratified × 3 repeat out-of-fold (OOF) değerlendirme yapar. Her repeat içinde her soru yalnız bir test fold'unda bulunur. Model kimliği ve `manual_is_correct` hiçbir zaman predictor değildir. `reference_answer`, alias, raw answer veya manual error type okunmaz.

Stage 05'te iki başarısız confidence ölçümü için *yeniden inference / gerektiğinde alternatif prompt* kullanılmıştır. Bu sürüm yalnız `repair1` olarak izlenir; orijinal inference ile özdeş olduğu iddia edilmez. Repair ham kayıtlarının karşılaştırılması bu notebook'un dışında, sonraki sensitivity audit'te yapılır.

Önceden tanımlanmış yöntemler: train-fold prevalence, raw confidence baselines, verification üzerinde Platt/logistic ve isotonic, tek-sinyal/feature-family logistic, CORE, EXTENDED, CORE ablation. **Tüm yöntemlerde aynı fold atamaları kullanılır.**

Önemli: Stage 09'daki single-signal metrikleri yöntem seçimi için keşif amaçlı kullanıldı. Buradaki OOF sonuçları da development sonuçlarıdır; sonradan seçilecek yöntemin nihai genelleme performansı bağımsız, önceden ayrılmış test setinde ölçülmelidir.

In [1]:
from google.colab import drive
drive.mount("/content/drive")
!pip -q install "scikit-learn>=1.5,<1.9" "pyarrow>=15,<25"

Mounted at /content/drive


## 1. Repair-aware frozen input ve provenance

In [2]:
from pathlib import Path
import json, hashlib, os, math
import numpy as np
import pandas as pd
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss

ROOT=Path('/content/drive/MyDrive/LLM_Confidence_Thesis')
VERSION='validated_v007'
ASSEMBLY='feature_assembly_v2_repair1'
BASE=ROOT/'features'/'assembled'/VERSION
M08=BASE/'stage08_manifest_v2_repair1.json'
SCHEMA=BASE/'feature_schema_v1.json'
MATRIX=BASE/'extended_model_matrix_v2_repair1.parquet'
M09=ROOT/'results'/'stage09'/f'{VERSION}_stage09_exploratory_manifest.json'
REPAIR=ROOT/'results'/'stage05'/f'{VERSION}_stage05_self_confidence_repair_v1.json'

def sha_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()

def load_json(path):
    with open(path,encoding='utf-8') as f:return json.load(f)

for path in [M08,SCHEMA,MATRIX,M09,REPAIR]:
    if not path.is_file():raise FileNotFoundError(path)
m08=load_json(M08);s=load_json(SCHEMA);m09=load_json(M09);r=load_json(REPAIR)
assert m08['dataset_version']==VERSION and m08['assembly_version']==ASSEMBLY
assert s['dataset_version']==VERSION and s['assembly_version']==ASSEMBLY
assert m09['stage08_manifest_sha256']==sha_file(M08)
assert m09['stage08_assembly_version']==ASSEMBLY and m09['stage05_repair_used'] is True
assert sha_file(MATRIX)==m08['outputs']['extended_model_matrix_sha256']
assert sha_file(SCHEMA)==m08['outputs']['feature_schema_file_sha256']
assert s['schema_sha256']==m08['feature_schema_sha256']
assert sha_file(REPAIR)==m08['upstream']['stage05_self_confidence_repair_manifest_sha256']
assert r['base_stage05_completion_sha256']==m08['upstream']['stage05_completion_sha256']
sidecar=Path(r['repair_file'])
assert sidecar.is_file() and sha_file(sidecar)==r['repair_file_sha256']
assert sha_file(sidecar)==m08['upstream']['stage05_self_confidence_repair_file_sha256']

full=pd.read_parquet(MATRIX)
core=list(s['core_features']);extended=list(s['extended_features']);target=s['target']
assert len(full)==3000 and full['model_id'].nunique()==3
assert not full.duplicated(['question_id','model_id']).any()
assert set(full[target].unique())=={0,1}
assert target not in extended and 'manual_error_type' not in extended
assert len(extended)==27 and len(core)==8
for f in extended:
    if f not in full:raise RuntimeError(f'Missing predictor: {f}')
    if not np.isfinite(pd.to_numeric(full[f],errors='coerce').to_numpy(dtype=float)).all():
        raise RuntimeError(f'Non-finite predictor: {f}')
clean=full.loc[full['exclude_from_final_test'].astype(int)==0].copy()
assert len(clean)==2910
assert (clean.groupby('model_id').size()==970).all()
assert (clean.groupby('model_id')['question_id'].nunique()==970).all()
assert not clean['used_in_model_selection'].astype(bool).any()
print('Stage08 repair-aware SHA/provenance: PASS')
print('Stage09 linked: PASS')
print('CLEAN_DEV 970 × 3 =',len(clean))
print('CORE:',len(core),'EXTENDED:',len(extended))
print('Repair manifest:', REPAIR)

Stage08 repair-aware SHA/provenance: PASS
Stage09 linked: PASS
CLEAN_DEV 970 × 3 = 2910
CORE: 8 EXTENDED: 27
Repair manifest: /content/drive/MyDrive/LLM_Confidence_Thesis/results/stage05/validated_v007_stage05_self_confidence_repair_v1.json


## 2. Önceden belirlenmiş yöntemler ve split sözleşmesi

In [3]:
SEEDS=[1101,2202,3303]
N_SPLITS=5
C_VALUE=1.0
EPS=1e-6
ECE_BINS=10

# Tek sinyal ve feature-family ablation: test performansına göre feature seçilmez.
INTERNAL=['mean_logprob','min_logprob','mean_token_entropy_normalized']
BEHAVIORAL=['self_confidence_scaled','verification_p_correct','self_consistency','unknown_response']

LOGISTIC_FEATURES={
    'platt_verification':['verification_p_correct'],
    'logistic_self_confidence':['self_confidence_scaled'],
    'logistic_consistency':['self_consistency'],
    'internal_only':INTERNAL,
    'behavioral_only':BEHAVIORAL,
    'core':core,
    'extended':extended,
    'core_minus_verification':[f for f in core if f!='verification_p_correct'],
    'core_minus_self_confidence':[f for f in core if f!='self_confidence_scaled'],
    'core_minus_consistency':[f for f in core if f!='self_consistency'],
    'core_minus_internal':[f for f in core if f not in INTERNAL],
}
RAW_SCORES={
    'raw_verification':'verification_p_correct',
    'raw_self_confidence':'self_confidence_scaled',
    'raw_consistency':'self_consistency',
}
ALL_METHODS=(['train_prevalence']+list(RAW_SCORES)+['isotonic_verification']+list(LOGISTIC_FEATURES))
assert len(ALL_METHODS)==len(set(ALL_METHODS))
assert all(set(fs).issubset(set(extended)) for fs in LOGISTIC_FEATURES.values())
print('Methods:',len(ALL_METHODS),ALL_METHODS)
print('Split contract:',N_SPLITS,'folds ×',len(SEEDS),'repeats')

Methods: 16 ['train_prevalence', 'raw_verification', 'raw_self_confidence', 'raw_consistency', 'isotonic_verification', 'platt_verification', 'logistic_self_confidence', 'logistic_consistency', 'internal_only', 'behavioral_only', 'core', 'extended', 'core_minus_verification', 'core_minus_self_confidence', 'core_minus_consistency', 'core_minus_internal']
Split contract: 5 folds × 3 repeats


## 3. Model başına fixed/reusable CV splits

In [4]:
splits={}
fold_audit=[]
for model_id, part in clean.groupby('model_id',sort=True):
    part=part.sort_values('question_id').reset_index(drop=True)
    y=part[target].astype(int).to_numpy()
    assert len(np.unique(y))==2
    for rep,seed in enumerate(SEEDS):
        cv=RepeatedStratifiedKFold(n_splits=N_SPLITS,n_repeats=1,random_state=seed)
        visited=np.zeros(len(part),dtype=int)
        for fold,(tr,te) in enumerate(cv.split(np.zeros(len(part)),y)):
            assert not np.intersect1d(tr,te).size
            assert len(np.unique(y[tr]))==len(np.unique(y[te]))==2
            visited[te]+=1
            splits[(model_id,rep,fold)]=(tr,te)
            fold_audit.append({'model_id':model_id,'repeat':rep,'fold':fold,'n_train':len(tr),'n_test':len(te),
                               'train_correct':int(y[tr].sum()),'test_correct':int(y[te].sum()),
                               'train_question_overlap_test':int(np.intersect1d(part.iloc[tr]['question_id'],part.iloc[te]['question_id']).size)})
        assert np.all(visited==1)
fold_df=pd.DataFrame(fold_audit)
display(fold_df.groupby('model_id').agg(n_folds=('fold','count'),min_test=('n_test','min'),max_test=('n_test','max'),max_question_overlap=('train_question_overlap_test','max')))
print('LEAKAGE-FREE FOLD ASSIGNMENTS: PASS')

,n_folds,min_test,max_test,max_question_overlap
model_id,,,,
Qwen/Qwen3-4B,15,194,194,0
google/gemma-3-4b-it,15,194,194,0
microsoft/Phi-4-mini-instruct,15,194,194,0


LEAKAGE-FREE FOLD ASSIGNMENTS: PASS


## 4. Train-only logistic/isotonic fit ve OOF tahminleri

In [5]:
def fit_predict(method,train,test,y_train):
    if method=='train_prevalence':return np.full(len(test),float(y_train.mean()))
    if method in RAW_SCORES:
        return pd.to_numeric(test[RAW_SCORES[method]],errors='raise').to_numpy(dtype=float)
    if method=='isotonic_verification':
        fit=IsotonicRegression(y_min=0.,y_max=1.,increasing=True,out_of_bounds='clip')
        fit.fit(train['verification_p_correct'].to_numpy(dtype=float),y_train)
        return fit.predict(test['verification_p_correct'].to_numpy(dtype=float))
    features=LOGISTIC_FEATURES[method]
    pipe=Pipeline([
        ('scaler',StandardScaler()),
        ('logistic',LogisticRegression(C=C_VALUE,solver='lbfgs',max_iter=3000,random_state=17)),
    ])
    pipe.fit(train[features].to_numpy(dtype=float),y_train)
    return pipe.predict_proba(test[features].to_numpy(dtype=float))[:,1]

prediction_rows=[]
for model_id,part in clean.groupby('model_id',sort=True):
    part=part.sort_values('question_id').reset_index(drop=True)
    yy=part[target].astype(int).to_numpy()
    for rep in range(len(SEEDS)):
        for fold in range(N_SPLITS):
            tr,te=splits[(model_id,rep,fold)]
            training=part.iloc[tr]
            heldout=part.iloc[te]
            for method in ALL_METHODS:
                pred=np.asarray(fit_predict(method,training,heldout,yy[tr]),dtype=float)
                if pred.shape!=(len(te),) or not np.isfinite(pred).all() or (pred < -1e-9).any() or (pred > 1+1e-9).any():
                    raise RuntimeError(f'Invalid predictions: {model_id} {rep} {fold} {method}')
                pred=np.clip(pred,0,1)
                for idx,prob in zip(te,pred):
                    prediction_rows.append({'model_id':model_id,'question_id':part.loc[idx,'question_id'],
                        'repeat':rep,'fold':fold,'method':method,
                        'manual_is_correct':int(yy[idx]),'p_correct':float(prob)})
    print(model_id,'completed')
oof=pd.DataFrame(prediction_rows)
assert len(oof)==len(clean)*len(SEEDS)*len(ALL_METHODS)
assert not oof.duplicated(['model_id','question_id','repeat','method']).any()
assert (oof.groupby(['model_id','repeat','method']).size()==970).all()
# Per repeat, all methods must be on exactly same held-out questions and fold assignments.
assert (oof.groupby(['model_id','question_id','repeat'])['fold'].nunique()==1).all()
assert (oof.groupby(['model_id','question_id','repeat'])['manual_is_correct'].nunique()==1).all()
print('OOF COVERAGE + SAME-FOLD CONTRACT: PASS',oof.shape)

Qwen/Qwen3-4B completed
google/gemma-3-4b-it completed
microsoft/Phi-4-mini-instruct completed
OOF COVERAGE + SAME-FOLD CONTRACT: PASS (139680, 7)


## 5. Değerlendirme metrikleri

Her repeat ayrı 970 benzersiz OOF tahmininden değerlendirilir. Repeat'ler farklı eğitim setleri kullandığı için aynı sorunun 3 tahmini **bağımsız 2910 örnek gibi sayılmaz**. `ECE=10` sabit-width bin kullanır; küçük bin'lerin örnek sayıları ayrıca kaydedilir. NLL için `EPS=1e-6` önceden tanımlanmış numerical clipping uygulanır. AUROC/AUPRC discrimination; Brier/NLL/ECE calibration/probability-quality diagnostic'tir. AUPRC prevalence'a bağlıdır.

In [6]:
def calc_ece(y,p,bins=ECE_BINS):
    idx=np.minimum((np.clip(p,0,1)*bins).astype(int),bins-1)
    table=[];ece=0.
    for b in range(bins):
        mask=idx==b;n=int(mask.sum())
        if not n:continue
        conf=float(p[mask].mean());acc=float(y[mask].mean());gap=abs(conf-acc)
        ece+=(n/len(y))*gap
        table.append({'bin':b,'n':n,'mean_confidence':conf,'empirical_accuracy':acc,'absolute_gap':gap})
    return float(ece),table

def calc_metrics(y,p):
    y=np.asarray(y,dtype=int);p=np.asarray(p,dtype=float)
    pclip=np.clip(p,EPS,1-EPS)
    ece,bins=calc_ece(y,p)
    return {'n':len(y),'n_correct':int(y.sum()),'prevalence':float(y.mean()),
        'auroc':float(roc_auc_score(y,p)),'auprc':float(average_precision_score(y,p)),
        'brier':float(brier_score_loss(y,p)),
        'nll':float(log_loss(y,pclip,labels=[0,1])),
        'ece_10_bins':ece,'mean_predicted_confidence':float(p.mean()),
        'mean_confidence_minus_accuracy':float(p.mean()-y.mean()),'reliability_bins':bins}

metric_rows=[];reliability_rows=[]
for (model_id,rep,method),part in oof.groupby(['model_id','repeat','method'],sort=True):
    part=part.sort_values('question_id')
    m=calc_metrics(part['manual_is_correct'],part['p_correct'])
    bins=m.pop('reliability_bins')
    metric_rows.append({'model_id':model_id,'repeat':int(rep),'method':method,**m})
    for b in bins:reliability_rows.append({'model_id':model_id,'repeat':int(rep),'method':method,**b})
metrics=pd.DataFrame(metric_rows);reliability=pd.DataFrame(reliability_rows)
print('Repeated OOF metric rows:',len(metrics))
display(metrics.query('repeat==0')[['model_id','method','n','prevalence','auroc','auprc','brier','nll','ece_10_bins']])

Repeated OOF metric rows: 144


,model_id,method,n,prevalence,auroc,auprc,brier,nll,ece_10_bins
0,Qwen/Qwen3-4B,behavioral_only,970,0.469072,0.899597,0.857930,0.120878,0.388261,2.328308e-02
1,Qwen/Qwen3-4B,core,970,0.469072,0.909071,0.863400,0.113931,0.369332,2.207811e-02
2,Qwen/Qwen3-4B,core_minus_consistency,970,0.469072,0.909173,0.864185,0.113985,0.369230,2.573283e-02
3,Qwen/Qwen3-4B,core_minus_internal,970,0.469072,0.901905,0.858331,0.119614,0.385401,2.077143e-02
4,Qwen/Qwen3-4B,core_minus_self_confidence,970,0.469072,0.905413,0.861556,0.117803,0.381072,3.062481e-02
5,Qwen/Qwen3-4B,core_minus_verification,970,0.469072,0.900169,0.850598,0.121460,0.388060,2.952833e-02
6,Qwen/Qwen3-4B,extended,970,0.469072,0.904573,0.856122,0.117108,0.380073,4.336639e-02
7,Qwen/Qwen3-4B,internal_only,970,0.469072,0.770993,0.700237,0.192506,0.565000,3.029103e-02
8,Qwen/Qwen3-4B,isotonic_verification,970,0.469072,0.888471,0.878594,0.129321,0.431943,3.361249e-02
9,Qwen/Qwen3-4B,logistic_consistency,970,0.469072,0.701626,0.605332,0.207313,0.600562,1.738658e-02


## 6. Model başına repeated OOF özetleri; train prevalence ile eşleştirilmiş farklar

In [7]:
summary=(metrics.groupby(['model_id','method'],as_index=False)
    .agg(n_repeats=('repeat','nunique'),mean_auroc=('auroc','mean'),sd_auroc=('auroc','std'),
         mean_auprc=('auprc','mean'),mean_brier=('brier','mean'),sd_brier=('brier','std'),
         mean_nll=('nll','mean'),mean_ece_10_bins=('ece_10_bins','mean'),mean_confidence=('mean_predicted_confidence','mean')))
reference=(metrics.loc[metrics['method']=='train_prevalence',
    ['model_id','repeat','brier','nll','ece_10_bins']].rename(columns={'brier':'base_brier','nll':'base_nll','ece_10_bins':'base_ece'}))
paired=metrics.merge(reference,on=['model_id','repeat'],validate='many_to_one')
paired['delta_brier_vs_train_prevalence']=paired['brier']-paired['base_brier']
paired['delta_nll_vs_train_prevalence']=paired['nll']-paired['base_nll']
paired['delta_ece_vs_train_prevalence']=paired['ece_10_bins']-paired['base_ece']
deltas=(paired.groupby(['model_id','method'],as_index=False)
    .agg(mean_delta_brier=('delta_brier_vs_train_prevalence','mean'),
         mean_delta_nll=('delta_nll_vs_train_prevalence','mean'),
         mean_delta_ece=('delta_ece_vs_train_prevalence','mean')))
summary=summary.merge(deltas,on=['model_id','method'],validate='one_to_one')
print('Negative delta: lower loss than train-fold prevalence baseline.')
display(summary[['model_id','method','n_repeats','mean_auroc','mean_auprc','mean_brier','mean_delta_brier','mean_nll','mean_delta_nll','mean_ece_10_bins']])

Negative delta: lower loss than train-fold prevalence baseline.


,model_id,method,n_repeats,mean_auroc,mean_auprc,mean_brier,mean_delta_brier,mean_nll,mean_delta_nll,mean_ece_10_bins
0,Qwen/Qwen3-4B,behavioral_only,3,0.902371,0.862544,0.120516,-0.128527,0.386643,-0.304590,2.271619e-02
1,Qwen/Qwen3-4B,core,3,0.910633,0.870515,0.113458,-0.135585,0.367359,-0.323874,2.210713e-02
2,Qwen/Qwen3-4B,core_minus_consistency,3,0.910921,0.871582,0.113452,-0.135591,0.366984,-0.324249,2.223233e-02
3,Qwen/Qwen3-4B,core_minus_internal,3,0.904570,0.866805,0.119082,-0.129961,0.383480,-0.307753,1.971060e-02
4,Qwen/Qwen3-4B,core_minus_self_confidence,3,0.907027,0.868074,0.117423,-0.131620,0.379277,-0.311956,2.966735e-02
5,Qwen/Qwen3-4B,core_minus_verification,3,0.900914,0.856893,0.121338,-0.127706,0.387656,-0.303577,2.272321e-02
6,Qwen/Qwen3-4B,extended,3,0.906106,0.863118,0.116306,-0.132737,0.377785,-0.313448,3.081448e-02
7,Qwen/Qwen3-4B,internal_only,3,0.773868,0.714121,0.192230,-0.056813,0.564604,-0.126629,4.059330e-02
8,Qwen/Qwen3-4B,isotonic_verification,3,0.891025,0.882356,0.127332,-0.121711,0.424640,-0.266593,2.883327e-02
9,Qwen/Qwen3-4B,logistic_consistency,3,0.704480,0.608511,0.207381,-0.041663,0.600723,-0.090510,1.748715e-02


In [8]:

# ============================================================
# STAGE 10 — ADDITIONAL AUDIT 1
# Fold-level Platt verification audit
# ============================================================

from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

platt_fold_rows = []
platt_pooled_rows = []

for model_id, part in clean.groupby("model_id", sort=True):

    part = (
        part.sort_values("question_id")
        .reset_index(drop=True)
    )

    y = part[target].astype(int).to_numpy()

    for rep in range(len(SEEDS)):

        for fold in range(N_SPLITS):

            tr, te = splits[(model_id, rep, fold)]

            training = part.iloc[tr]
            heldout = part.iloc[te]

            x_train = training[
                ["verification_p_correct"]
            ].to_numpy(dtype=float)

            x_test = heldout[
                ["verification_p_correct"]
            ].to_numpy(dtype=float)

            y_train = y[tr]
            y_test = y[te]

            # Aynı Stage 10 preprocessing + Platt modeli.
            pipe = Pipeline([
                ("scaler", StandardScaler()),
                (
                    "logistic",
                    LogisticRegression(
                        C=C_VALUE,
                        solver="lbfgs",
                        max_iter=3000,
                        random_state=17,
                    ),
                ),
            ])

            pipe.fit(x_train, y_train)

            raw_scores = x_test[:, 0]

            calibrated_scores = (
                pipe.predict_proba(x_test)[:, 1]
            )

            scaler = pipe.named_steps["scaler"]
            logistic = pipe.named_steps["logistic"]

            standardized_coef = float(
                logistic.coef_[0, 0]
            )

            # Katsayının ham verification ölçeğindeki karşılığı.
            raw_scale_coef = (
                standardized_coef
                / float(scaler.scale_[0])
            )

            raw_auroc = float(
                roc_auc_score(y_test, raw_scores)
            )

            platt_auroc = float(
                roc_auc_score(
                    y_test,
                    calibrated_scores,
                )
            )

            # Yeniden fit edilen tahminler ile orijinal
            # Stage 10 OOF tahminlerini karşılaştır.
            saved = oof.loc[
                (oof["model_id"] == model_id)
                & (oof["repeat"] == rep)
                & (oof["fold"] == fold)
                & (oof["method"] == "platt_verification"),
                ["question_id", "p_correct"],
            ]

            comparison = (
                heldout[["question_id"]]
                .assign(
                    refit_prediction=calibrated_scores
                )
                .merge(
                    saved,
                    on="question_id",
                    how="inner",
                    validate="one_to_one",
                )
            )

            assert len(comparison) == len(te)

            max_prediction_difference = float(
                np.max(
                    np.abs(
                        comparison["refit_prediction"]
                        - comparison["p_correct"]
                    )
                )
            )

            if max_prediction_difference > 1e-8:
                raise RuntimeError(
                    "Refit ve orijinal OOF tahminleri "
                    "eşleşmiyor: "
                    f"{model_id}, rep={rep}, fold={fold}"
                )

            platt_fold_rows.append({
                "model_id": model_id,
                "repeat": rep,
                "fold": fold,
                "n_test": len(te),

                "standardized_coef": standardized_coef,
                "raw_scale_coef": raw_scale_coef,

                "coef_positive": raw_scale_coef > 0,
                "coef_negative": raw_scale_coef < 0,

                "raw_fold_auroc": raw_auroc,
                "platt_fold_auroc": platt_auroc,

                "delta_fold_auroc": (
                    platt_auroc - raw_auroc
                ),

                "platt_min_probability": float(
                    calibrated_scores.min()
                ),

                "platt_max_probability": float(
                    calibrated_scores.max()
                ),

                "max_refit_oof_difference": (
                    max_prediction_difference
                ),
            })

        # Beş fold'un birleştirilmiş OOF AUROC'u.
        raw_rep = (
            oof.loc[
                (oof["model_id"] == model_id)
                & (oof["repeat"] == rep)
                & (oof["method"] == "raw_verification")
            ]
            .sort_values("question_id")
        )

        platt_rep = (
            oof.loc[
                (oof["model_id"] == model_id)
                & (oof["repeat"] == rep)
                & (oof["method"] == "platt_verification")
            ]
            .sort_values("question_id")
        )

        assert (
            raw_rep["question_id"].tolist()
            == platt_rep["question_id"].tolist()
        )

        assert (
            raw_rep["manual_is_correct"].tolist()
            == platt_rep["manual_is_correct"].tolist()
        )

        pooled_y = (
            raw_rep["manual_is_correct"]
            .astype(int)
            .to_numpy()
        )

        pooled_raw_auc = float(
            roc_auc_score(
                pooled_y,
                raw_rep["p_correct"].to_numpy(),
            )
        )

        pooled_platt_auc = float(
            roc_auc_score(
                pooled_y,
                platt_rep["p_correct"].to_numpy(),
            )
        )

        platt_pooled_rows.append({
            "model_id": model_id,
            "repeat": rep,
            "n": len(raw_rep),
            "pooled_raw_auroc": pooled_raw_auc,
            "pooled_platt_auroc": pooled_platt_auc,
            "pooled_auroc_difference": (
                pooled_platt_auc - pooled_raw_auc
            ),
        })


platt_fold_audit = pd.DataFrame(platt_fold_rows)
platt_pooled_audit = pd.DataFrame(platt_pooled_rows)

# ============================================================
# Kontrol 1: Katsayı işareti ve fold içi AUROC farkı
# ============================================================

platt_fold_summary = (
    platt_fold_audit
    .groupby("model_id", as_index=False)
    .agg(
        n_folds=("fold", "count"),
        n_positive_coef=("coef_positive", "sum"),
        n_negative_coef=("coef_negative", "sum"),

        min_raw_scale_coef=("raw_scale_coef", "min"),
        max_raw_scale_coef=("raw_scale_coef", "max"),

        mean_raw_fold_auroc=("raw_fold_auroc", "mean"),
        mean_platt_fold_auroc=("platt_fold_auroc", "mean"),

        max_abs_fold_auroc_difference=(
            "delta_fold_auroc",
            lambda x: float(np.abs(x).max()),
        ),

        max_refit_oof_difference=(
            "max_refit_oof_difference",
            "max",
        ),
    )
)

print("1 — FOLD-LEVEL PLATT AUDIT")
display(platt_fold_summary)


# ============================================================
# Kontrol 2: Birleştirilmiş OOF AUROC
# ============================================================

print("2 — POOLED OOF AUROC")
display(platt_pooled_audit)


# ============================================================
# Kontrol 3: Gemma'nın 15 fold'unu ayrıntılı göster
# ============================================================

gemma_platt_audit = (
    platt_fold_audit.loc[
        platt_fold_audit["model_id"]
        == "google/gemma-3-4b-it"
    ]
    .sort_values(["repeat", "fold"])
)

print("3 — GEMMA: HER FOLD")
display(
    gemma_platt_audit[
        [
            "repeat",
            "fold",
            "standardized_coef",
            "raw_scale_coef",
            "raw_fold_auroc",
            "platt_fold_auroc",
            "delta_fold_auroc",
            "platt_min_probability",
            "platt_max_probability",
        ]
    ]
)

print("PLATT FOLD AUDIT: COMPLETE")

1 — FOLD-LEVEL PLATT AUDIT


,model_id,n_folds,n_positive_coef,n_negative_coef,min_raw_scale_coef,max_raw_scale_coef,mean_raw_fold_auroc,mean_platt_fold_auroc,max_abs_fold_auroc_difference,max_refit_oof_difference
0,Qwen/Qwen3-4B,15,15,0,3.025251,3.342441,0.898851,0.898851,1.110223e-16,0.0
1,google/gemma-3-4b-it,15,15,0,1.760718,2.072705,0.782398,0.782398,0.000000e+00,0.0
2,microsoft/Phi-4-mini-instruct,15,15,0,2.930389,3.367466,0.820718,0.820718,0.000000e+00,0.0


2 — POOLED OOF AUROC


,model_id,repeat,n,pooled_raw_auroc,pooled_platt_auroc,pooled_auroc_difference
0,Qwen/Qwen3-4B,0,970,0.898671,0.840243,-0.058427
1,Qwen/Qwen3-4B,1,970,0.898671,0.842189,-0.056481
2,Qwen/Qwen3-4B,2,970,0.898671,0.845535,-0.053136
3,google/gemma-3-4b-it,0,970,0.782700,0.653745,-0.128954
4,google/gemma-3-4b-it,1,970,0.782700,0.665816,-0.116884
5,google/gemma-3-4b-it,2,970,0.782700,0.655655,-0.127044
6,microsoft/Phi-4-mini-instruct,0,970,0.820579,0.806972,-0.013607
7,microsoft/Phi-4-mini-instruct,1,970,0.820579,0.811837,-0.008742
8,microsoft/Phi-4-mini-instruct,2,970,0.820579,0.809075,-0.011504


3 — GEMMA: HER FOLD


,repeat,fold,standardized_coef,raw_scale_coef,raw_fold_auroc,platt_fold_auroc,delta_fold_auroc,platt_min_probability,platt_max_probability
15,0,0,0.583519,1.760718,0.845881,0.845881,0.0,0.298796,0.712520
16,0,1,0.689545,1.957243,0.789654,0.789654,0.0,0.272887,0.726552
17,0,2,0.712112,2.072705,0.715554,0.715554,0.0,0.250032,0.725964
18,0,3,0.698941,2.034159,0.791430,0.791430,0.0,0.256434,0.725031
19,0,4,0.696432,2.067723,0.775649,0.775649,0.0,0.249345,0.724243
20,1,0,0.687243,1.991356,0.815814,0.815814,0.0,0.264114,0.724450
21,1,1,0.673511,1.986382,0.774621,0.774621,0.0,0.262596,0.721886
22,1,2,0.665056,1.959784,0.751184,0.751184,0.0,0.266704,0.720787
23,1,3,0.683968,1.972535,0.773674,0.773674,0.0,0.268324,0.724994
24,1,4,0.669153,1.980355,0.796568,0.796568,0.0,0.263977,0.722108


PLATT FOLD AUDIT: COMPLETE


In [9]:

# ============================================================
# STAGE 10 — ADDITIONAL AUDIT 2
# Paired question-level bootstrap for ablation
# ============================================================

BOOTSTRAP_REPS = 4000
BOOTSTRAP_SEED = 20260919

BASE_METHOD = "core"

COMPARISON_METHODS = [
    "extended",
    "core_minus_consistency",
    "core_minus_internal",
    "core_minus_verification",
    "core_minus_self_confidence",
    "behavioral_only",
]

paired_bootstrap_rows = []

# Mevcut Stage 10 NLL clipping sözleşmesini kullan.
NLL_EPS = EPS

for model_idx, (model_id, model_df) in enumerate(
    clean.groupby("model_id", sort=True)
):

    question_ids = sorted(
        model_df["question_id"].unique()
    )

    n_questions = len(question_ids)
    n_repeats = len(SEEDS)

    assert n_questions == 970

    # Her soru x tekrar için sabit satır sırası.
    expected_index = pd.MultiIndex.from_product(
        [
            question_ids,
            range(n_repeats),
        ],
        names=[
            "question_id",
            "repeat",
        ],
    )

    def get_prediction_matrix(method):

        selected = oof.loc[
            (oof["model_id"] == model_id)
            & (oof["method"] == method),
            [
                "question_id",
                "repeat",
                "manual_is_correct",
                "p_correct",
            ],
        ].copy()

        if selected.duplicated(
            ["question_id", "repeat"]
        ).any():
            raise RuntimeError(
                f"Duplicate OOF rows: {model_id} / {method}"
            )

        selected = selected.set_index(
            ["question_id", "repeat"]
        )

        if set(selected.index) != set(expected_index):
            raise RuntimeError(
                f"OOF coverage mismatch: {model_id} / {method}"
            )

        selected = selected.loc[expected_index]

        probabilities = (
            selected["p_correct"]
            .to_numpy(dtype=float)
            .reshape(n_questions, n_repeats)
        )

        labels = (
            selected["manual_is_correct"]
            .to_numpy(dtype=int)
            .reshape(n_questions, n_repeats)
        )

        if not np.isfinite(probabilities).all():
            raise RuntimeError(
                f"Nonfinite prediction: {model_id} / {method}"
            )

        return probabilities, labels

    core_p, core_y = get_prediction_matrix(
        BASE_METHOD
    )

    # Bir sorunun manuel doğruluk etiketi her tekrarda aynı.
    assert np.all(
        core_y == core_y[:, [0]]
    )

    # Bootstrap birimi: question_id.
    #
    # Aynı sorunun üç tekrarındaki tahminler birlikte seçilecek.
    rng = np.random.default_rng(
        BOOTSTRAP_SEED + model_idx
    )

    bootstrap_question_indices = rng.integers(
        0,
        n_questions,
        size=(
            BOOTSTRAP_REPS,
            n_questions,
        ),
    )

    # CORE için soru x tekrar kayıp matrisi.
    core_brier_loss = (
        core_p - core_y
    ) ** 2

    core_p_clipped = np.clip(
        core_p,
        NLL_EPS,
        1.0 - NLL_EPS,
    )

    core_nll_loss = -(
        core_y * np.log(core_p_clipped)
        +
        (1 - core_y)
        * np.log(1 - core_p_clipped)
    )

    for alternative in COMPARISON_METHODS:

        alt_p, alt_y = get_prediction_matrix(
            alternative
        )

        if not np.array_equal(core_y, alt_y):
            raise RuntimeError(
                "Method label mismatch: "
                f"{model_id} / {alternative}"
            )

        alt_brier_loss = (
            alt_p - alt_y
        ) ** 2

        alt_p_clipped = np.clip(
            alt_p,
            NLL_EPS,
            1.0 - NLL_EPS,
        )

        alt_nll_loss = -(
            alt_y * np.log(alt_p_clipped)
            +
            (1 - alt_y)
            * np.log(1 - alt_p_clipped)
        )

        # Alternatif - CORE:
        # Pozitif fark CORE'un daha düşük hata verdiğini gösterir.
        delta_brier_matrix = (
            alt_brier_loss - core_brier_loss
        )

        delta_nll_matrix = (
            alt_nll_loss - core_nll_loss
        )

        # Bir sorunun üç tekrarındaki kayıp farkının ortalaması.
        per_question_delta_brier = (
            delta_brier_matrix.mean(axis=1)
        )

        per_question_delta_nll = (
            delta_nll_matrix.mean(axis=1)
        )

        observed_delta_brier = float(
            per_question_delta_brier.mean()
        )

        observed_delta_nll = float(
            per_question_delta_nll.mean()
        )

        # Question-level paired bootstrap.
        boot_brier = (
            per_question_delta_brier[
                bootstrap_question_indices
            ]
            .mean(axis=1)
        )

        boot_nll = (
            per_question_delta_nll[
                bootstrap_question_indices
            ]
            .mean(axis=1)
        )

        brier_ci_low, brier_ci_high = np.percentile(
            boot_brier,
            [2.5, 97.5],
        )

        nll_ci_low, nll_ci_high = np.percentile(
            boot_nll,
            [2.5, 97.5],
        )

        # Üç tekrarın ayrı ayrı ortalama farkları.
        repeat_brier_deltas = (
            delta_brier_matrix.mean(axis=0)
        )

        repeat_nll_deltas = (
            delta_nll_matrix.mean(axis=0)
        )

        paired_bootstrap_rows.append({
            "model_id": model_id,
            "base_method": BASE_METHOD,
            "alternative_method": alternative,

            "n_unique_questions": n_questions,
            "n_repeats": n_repeats,

            "delta_brier_alt_minus_core": (
                observed_delta_brier
            ),

            "delta_brier_ci95_low": float(
                brier_ci_low
            ),

            "delta_brier_ci95_high": float(
                brier_ci_high
            ),

            "delta_nll_alt_minus_core": (
                observed_delta_nll
            ),

            "delta_nll_ci95_low": float(
                nll_ci_low
            ),

            "delta_nll_ci95_high": float(
                nll_ci_high
            ),

            "repeat0_delta_brier": float(
                repeat_brier_deltas[0]
            ),

            "repeat1_delta_brier": float(
                repeat_brier_deltas[1]
            ),

            "repeat2_delta_brier": float(
                repeat_brier_deltas[2]
            ),

            "repeat0_delta_nll": float(
                repeat_nll_deltas[0]
            ),

            "repeat1_delta_nll": float(
                repeat_nll_deltas[1]
            ),

            "repeat2_delta_nll": float(
                repeat_nll_deltas[2]
            ),

            "bootstrap_reps": BOOTSTRAP_REPS,
        })


paired_bootstrap_audit = pd.DataFrame(
    paired_bootstrap_rows
)

print("PAIRED BOOTSTRAP — BRIER")

display(
    paired_bootstrap_audit[
        [
            "model_id",
            "alternative_method",
            "delta_brier_alt_minus_core",
            "delta_brier_ci95_low",
            "delta_brier_ci95_high",
            "repeat0_delta_brier",
            "repeat1_delta_brier",
            "repeat2_delta_brier",
        ]
    ]
)

print("PAIRED BOOTSTRAP — NLL")

display(
    paired_bootstrap_audit[
        [
            "model_id",
            "alternative_method",
            "delta_nll_alt_minus_core",
            "delta_nll_ci95_low",
            "delta_nll_ci95_high",
        ]
    ]
)

print("PAIRED QUESTION-LEVEL BOOTSTRAP: COMPLETE")

PAIRED BOOTSTRAP — BRIER


,model_id,alternative_method,delta_brier_alt_minus_core,delta_brier_ci95_low,delta_brier_ci95_high,repeat0_delta_brier,repeat1_delta_brier,repeat2_delta_brier
0,Qwen/Qwen3-4B,extended,0.002848,0.001218,0.004519,0.003177,0.002841,0.002527
1,Qwen/Qwen3-4B,core_minus_consistency,-0.000006,-0.000643,0.000590,0.000054,-0.000355,0.000283
2,Qwen/Qwen3-4B,core_minus_internal,0.005624,0.001972,0.009549,0.005683,0.005848,0.005342
3,Qwen/Qwen3-4B,core_minus_verification,0.007879,0.002441,0.013298,0.007528,0.008098,0.008011
4,Qwen/Qwen3-4B,core_minus_self_confidence,0.003965,0.000364,0.007740,0.003872,0.003982,0.004042
5,Qwen/Qwen3-4B,behavioral_only,0.007058,0.002735,0.011439,0.006947,0.007317,0.006911
6,google/gemma-3-4b-it,extended,0.001452,-0.002295,0.005542,0.001480,0.000783,0.002092
7,google/gemma-3-4b-it,core_minus_consistency,-0.000788,-0.001470,-0.000138,-0.000456,-0.001227,-0.000681
8,google/gemma-3-4b-it,core_minus_internal,0.021031,0.013562,0.028964,0.021034,0.020688,0.021370
9,google/gemma-3-4b-it,core_minus_verification,0.000781,-0.001786,0.003577,0.001098,0.000168,0.001077


PAIRED BOOTSTRAP — NLL


,model_id,alternative_method,delta_nll_alt_minus_core,delta_nll_ci95_low,delta_nll_ci95_high
0,Qwen/Qwen3-4B,extended,0.010426,0.005032,0.016121
1,Qwen/Qwen3-4B,core_minus_consistency,-0.000375,-0.002058,0.001277
2,Qwen/Qwen3-4B,core_minus_internal,0.016121,0.006192,0.026658
3,Qwen/Qwen3-4B,core_minus_verification,0.020298,0.006473,0.034111
4,Qwen/Qwen3-4B,core_minus_self_confidence,0.011919,0.002415,0.021940
5,Qwen/Qwen3-4B,behavioral_only,0.019284,0.007546,0.031119
6,google/gemma-3-4b-it,extended,0.013011,-0.001781,0.031048
7,google/gemma-3-4b-it,core_minus_consistency,-0.001739,-0.003261,-0.000283
8,google/gemma-3-4b-it,core_minus_internal,0.047773,0.025743,0.069729
9,google/gemma-3-4b-it,core_minus_verification,0.003303,-0.003400,0.010260


PAIRED QUESTION-LEVEL BOOTSTRAP: COMPLETE


In [10]:

# Main Stage10 immutable artifact'larından ayrı diagnostics klasörü.

DIAGNOSTICS_DIR = (
    ROOT
    / "results"
    / "stage10"
    / VERSION
    / ("stage08_" + sha_file(M08)[:12] + "_repair1")
    / "diagnostics_v1"
)

DIAGNOSTICS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

platt_fold_audit.to_csv(
    DIAGNOSTICS_DIR / "platt_fold_audit_v1.csv",
    index=False,
)

platt_fold_summary.to_csv(
    DIAGNOSTICS_DIR / "platt_fold_summary_v1.csv",
    index=False,
)

platt_pooled_audit.to_csv(
    DIAGNOSTICS_DIR / "platt_pooled_oof_audit_v1.csv",
    index=False,
)

paired_bootstrap_audit.to_csv(
    DIAGNOSTICS_DIR / "paired_ablation_bootstrap_v1.csv",
    index=False,
)

print("Additional Stage10 diagnostics saved:")
print(DIAGNOSTICS_DIR)

Additional Stage10 diagnostics saved:
/content/drive/MyDrive/LLM_Confidence_Thesis/results/stage10/validated_v007/stage08_a0bb8dfed269_repair1/diagnostics_v1


## 7. Stage 05 repair sensitivity — sınır

Bu notebook'un **ana** OOF değerlendirmesi repair-aware Stage08'i kullanır. İki yeniden üretilmiş confidence kaydı, orijinal tek-sorgu skoru ile aynı ölçüm protokolünden gelmez. Farklı prompt kullanılmışsa dağılım kayması da mümkündür. `repair1` provenance kayıtları manifestte tutulur.

Orijinal Stage05 sinyalleriyle eşleştirilmiş repair/no-repair karşılaştırması için aynı `question_id × model_id` üzerinde orijinal değerler, repair sonrası değerler, repair flag'i ve **aynı sabit CV split'leri** gerekir. Orijinal ölçümler non-numeric/parsing failed ise bu satırlar orijinal-skor tablosunda `missing` olarak açıkça raporlanmalıdır; bilinçsizce 0'a dönüştürülmemelidir. Bu veri olmadan bir repair etkisi metriği icat etmiyoruz. Böyle bir sensitivity analizi, ana OOF sonuçlarının yerini almaz.

## 8. Versioned immutable outputs / provenance

In [11]:
OUT=ROOT/'results'/'stage10'/VERSION/('stage08_'+sha_file(M08)[:12]+'_repair1')
OUT.mkdir(parents=True,exist_ok=True)
paths={
  'oof_predictions':OUT/'oof_predictions_v1.parquet',
  'per_repeat_metrics':OUT/'per_repeat_metrics_v1.csv',
  'summary':OUT/'repeated_oof_summary_v1.csv',
  'reliability_bins':OUT/'reliability_bins_v1.csv',
  'fold_audit':OUT/'fold_audit_v1.csv',
  'contract':OUT/'experiment_contract_v1.json',
  'manifest':OUT/'stage10_dev_manifest_v1.json',
}
contract={
 'stage':'10_posthoc_oof_dev','dataset_version':VERSION,'assembly_version':ASSEMBLY,
 'stage08_manifest_sha256':sha_file(M08),'stage09_manifest_sha256':sha_file(M09),
 'stage05_repair_manifest_sha256':sha_file(REPAIR),'stage05_repair_file_sha256':sha_file(sidecar),
 'stage05_repair_method_note':'two original measurement failures; new inference / optional alternative prompt; not merely parsing',
 'scope':'CLEAN_DEV: exclude_from_final_test==0','n_questions_per_model':970,'n_models':3,
 'n_splits':N_SPLITS,'seeds':SEEDS,'fixed_folds_shared_by_methods':True,
 'primary_method_contract':ALL_METHODS,'logistic_feature_contract':LOGISTIC_FEATURES,
 'raw_score_contract':RAW_SCORES,'logistic_C':C_VALUE,'logistic_standard_scaler':'fit_train_fold_only',
 'isotonic':'fit_train_fold_only','prevalence':'mean_train_fold_label_only',
 'nll_clip_eps':EPS,'ece_equal_width_bins':ECE_BINS,
 'assessment':'development_oof_not_independent_final_test',
 'repeat_evaluation':'one metric per repeat (970 unique OOF questions), report mean and sd; no 2910 independent samples',
}

def atomic_immutable_bytes(path,blob):
    if path.exists():
        if path.read_bytes()!=blob:raise RuntimeError(f'Immutable output content conflict: {path}')
        return
    temp=path.with_name(path.name+'.tmp')
    temp.write_bytes(blob);os.replace(temp,path)

def immutable_csv(df,path):
    atomic_immutable_bytes(path,df.to_csv(index=False).encode('utf-8'))

contract_blob=json.dumps(contract,ensure_ascii=False,sort_keys=True,indent=2).encode('utf-8')
atomic_immutable_bytes(paths['contract'],contract_blob)

# Save Parquet once; existing frozen result must match the content when read back.
if paths['oof_predictions'].exists():
    old=pd.read_parquet(paths['oof_predictions'])
    pd.testing.assert_frame_equal(old.reset_index(drop=True),oof.reset_index(drop=True),check_exact=True)
else:
    temp=paths['oof_predictions'].with_name(paths['oof_predictions'].name+'.tmp.parquet')
    oof.to_parquet(temp,index=False,engine='pyarrow');os.replace(temp,paths['oof_predictions'])
immutable_csv(metrics,paths['per_repeat_metrics'])
immutable_csv(summary,paths['summary'])
immutable_csv(reliability,paths['reliability_bins'])
immutable_csv(fold_df,paths['fold_audit'])
manifest={'contract_file':str(paths['contract']),'contract_sha256':sha_file(paths['contract']),
  'n_model_answer_rows':len(clean),'n_oof_prediction_rows':len(oof),'n_methods':len(ALL_METHODS),
  'n_repeats':len(SEEDS),'created_from_stage08_sha256':sha_file(M08),
  'outputs':{k:{'path':str(v),'sha256':sha_file(v)} for k,v in paths.items() if k!='manifest'}}
atomic_immutable_bytes(paths['manifest'],json.dumps(manifest,ensure_ascii=False,sort_keys=True,indent=2).encode('utf-8'))
print('STAGE 10 DEVELOPMENT OOF TAMAMLANDI')
print('Output:',OUT)
print('Manifest:',paths['manifest'])

STAGE 10 DEVELOPMENT OOF TAMAMLANDI
Output: /content/drive/MyDrive/LLM_Confidence_Thesis/results/stage10/validated_v007/stage08_a0bb8dfed269_repair1
Manifest: /content/drive/MyDrive/LLM_Confidence_Thesis/results/stage10/validated_v007/stage08_a0bb8dfed269_repair1/stage10_dev_manifest_v1.json


# Çıktıları değerlendirirken

1. Önce `train_prevalence`, `raw_verification`, `platt_verification`, `isotonic_verification`, `internal_only`, `behavioral_only`, `core`, `extended` için **her modelde** Brier/NLL/AUROC/AUPRC/ECE'yi inceleyin.
2. `core_minus_*` ablation'larında Brier/NLL değişimini, aynı fold'larda değerlendirilmiş `core` ile karşılaştırın. Çok yüksek korelasyon nedeniyle tek bir feature'ın çıkarılması küçük fark doğurabilir.
3. Bu OOF performansları **development** tahminleridir; modeller/feature setleri arasından en iyi görüneni burada seçip yine aynı OOF'de final performans ilan etmeyin.
4. Repair sidecar'ın orijinal ve yeni skorlarıyla eşleştirilmiş sensitivity analizi ayrı tamamlanmalıdır.
5. Nihai veri seti ve bağımsız test protokolü dondurulmadan final test sonucu sunulmaz.